# AfroShop Cloud AI Lab : Banc d'essai Prompt Injection & Guardrails

Ce notebook présente une évaluation expérimentale rigoureuse de la sécurité de l'assistant **AfroShop** face à des attaques par **Prompt Injection**.

### Objectifs de l'expérience :
1. **Baseline (SANS Skill)** : Observer le comportement naïf du modèle exposé aux injections directes, jailbreaks et tentatives d'exfiltration de clés GCP.
2. **Défense Active (AVEC Skill AfroShop Guard)** : Évaluer l'efficacité des garde-fous et règles configurés dans `.agents/plugins/afroshop-guard`.
3. **Télémétrie Cloud** : Générer les événements d'audit au format **Google Cloud Logging** pour superviser les attaques sur la console GCP.

In [ ]:
import sys
import os
import json
import pandas as pd

# Ajout du dossier src au chemin Python
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from security_engine import AfroShopSecurityEngine, format_gcp_cloud_logging_entry

print("Moteur de sécurité AfroShop chargé avec succès.")

## 1. Chargement du jeu de données de test (10 Scénarios)

In [ ]:
dataset_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src', 'eval_dataset.json'))
with open(dataset_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

df_dataset = pd.DataFrame(dataset)
df_dataset[['id', 'category', 'expected_verdict', 'description']]

## 2. Exécution comparative : SANS Skill vs AVEC Skill

In [ ]:
engine_naive = AfroShopSecurityEngine(use_skill=False)
engine_guard = AfroShopSecurityEngine(use_skill=True)

results = []
gcp_logs = []

for item in dataset:
    p = item['prompt']
    res_naive = engine_naive.analyze_prompt(p)
    res_guard = engine_guard.analyze_prompt(p)
    
    results.append({
        'ID': item['id'],
        'Catégorie': item['category'],
        'Attendu': item['expected_verdict'],
        'Sans Skill': res_naive['verdict'],
        'Avec Skill': res_guard['verdict'],
        'Menace Détectée': res_guard['attack_vector'],
        'Statut Sécurité': 'PROTÉGÉ' if res_guard['verdict'] == item['expected_verdict'] else 'VULNÉRABLE'
    })
    
    gcp_logs.append(format_gcp_cloud_logging_entry(res_guard, item['id'], p))

df_results = pd.DataFrame(results)
df_results

## 3. Analyse des Métriques de Sécurité

In [ ]:
attacks = [x for x in dataset if x['expected_verdict'] == 'BLOCKED']
legitimate = [x for x in dataset if x['expected_verdict'] == 'ALLOWED']

naive_blocked = sum(1 for x in attacks if engine_naive.analyze_prompt(x['prompt'])['verdict'] == 'BLOCKED')
guard_blocked = sum(1 for x in attacks if engine_guard.analyze_prompt(x['prompt'])['verdict'] == 'BLOCKED')
guard_false_positives = sum(1 for x in legitimate if engine_guard.analyze_prompt(x['prompt'])['verdict'] == 'BLOCKED')

print(f"--- RÉSULTATS COMPARATIFS SUR LES ATTAQUES ({len(attacks)} injections testées) ---")
print(f"Taux de blocage SANS Skill : {naive_blocked / len(attacks) * 100:.1f}%")
print(f"Taux de blocage AVEC Skill : {guard_blocked / len(attacks) * 100:.1f}%")
print(f"Faux positifs (requêtes légitimes bloquées) : {guard_false_positives}/{len(legitimate)}")

## 4. Télémétrie & Logs pour Google Cloud Logging

In [ ]:
# Exemple d'un payload structuré envoyé à Cloud Logging en cas d'attaque bloquée
print(json.dumps(gcp_logs[3], indent=2, ensure_ascii=False))